<a href="https://colab.research.google.com/github/julianeguo/exodiscovery-benchmark/blob/main/exodiscover_benchmark_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
import pandas as pd

df = pd.read_csv("part1_grouped_dataset.csv")
df.head()

In [ ]:
SYSTEM_PROMPT = """
You are given one group of anonymized exoplanets in table form.
Your task is to evaluate each planet independently and determine whether it is habitable.

The data includes the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets.
Base your decision strictly on the criteria above.

Return exactly 5 entries, one for each anon_id in the table.
Format each entry exactly as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After the 5th entry, write exactly:
END
Do not write anything after END.
"""

In [ ]:
def group_to_table(group_df):
    return group_df[["anon_id","pl_rade","pl_bmasse","pl_insol"]].to_string(index=False)

In [ ]:
def build_prompt(table):
    return SYSTEM_PROMPT + "\n\nDATA:\n" + table + "\n\nANSWER:\n"

def run_model(table):
    prompt = build_prompt(table)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=650,
        do_sample=False,
    )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]

    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print("\nFULL RESPONSE:")
    print(text)

    if "END" in text:
        text = text.split("END", 1)[0].strip() + "\nEND"

    return text

In [ ]:
results = []

groups = list(df.groupby("group_id"))
total_groups = len(groups)

for i, (gid, group) in enumerate(groups, 1):

    print(f"Running group {i}/{total_groups}")

    table = group_to_table(group)
    prompt = build_prompt(table)
    response = run_model(table)

    results.append({
        "group_id": gid,
        "prompt": prompt,
        "response": response
    })

    entry_count = response.count("anon_id:")
    if entry_count != 5:
      print(f"WARNING: group {gid} returned {entry_count} entries")

    print(f"Finished group {i}")

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("qwen_results_part1_final.csv", index=False)

In [ ]:
SYSTEM_PROMPT_PART_2 = """
You are given one anonymized exoplanet in table form.
Your task is to evaluate whether it is habitable.

The data will include the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Here is the evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets not present in the input.
Base your decision strictly on the numerical criteria above.

Return exactly 1 entry, formatted as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After answering, write exactly:
END
Do not write anything after END.
"""

In [ ]:
import pandas as pd

orig_df = pd.read_csv("borderline_12.csv")
orig_df.head()
mod_df = pd.read_csv("borderline_12_modified.csv")
mod_df.head()

In [ ]:
def row_to_table(row):
    return pd.DataFrame([row])[["anon_id", "pl_rade", "pl_bmasse", "pl_insol"]].to_string(index=False)

In [ ]:
def build_prompt_single(row):
    table = row_to_table(row)
    return "DATA:\n" + table + "\n\nANSWER:\n"

In [ ]:
def run_model_single(row):
    table = row_to_table(row)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_PART_2},
        {"role": "user", "content": "DATA:\n" + table + "\n\nANSWER:"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
    )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    marker = "END"
    if marker in text:
        text = text.split(marker, 1)[0].strip() + "\n" + marker

    return prompt, text

In [ ]:
results = []

total_pairs = len(orig_df)

for i in range(total_pairs):
    orig_row = orig_df.iloc[i]
    mod_row = mod_df.iloc[i]

    pair_id = orig_row["pair_id"]
    print(f"Running pair {i+1}/{total_pairs} (pair_id={pair_id})")

    # Original first
    orig_prompt, orig_response = run_model_single(orig_row)
    print("Original response:")
    print(orig_response)
    print("-" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "original",
        "anon_id": orig_row["anon_id"],
        "pl_name": orig_row["pl_name"],
        "pl_rade": orig_row["pl_rade"],
        "pl_bmasse": orig_row["pl_bmasse"],
        "pl_insol": orig_row["pl_insol"],
        "selected_threshold": orig_row["selected_threshold"],
        "label_habitable": orig_row["label_habitable"],
        "prompt": orig_prompt,
        "response": orig_response,
    })

    # Modified second
    mod_prompt, mod_response = run_model_single(mod_row)
    print("Modified response:")
    print(mod_response)
    print("=" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "modified",
        "anon_id": mod_row["anon_id"],
        "pl_name": mod_row["pl_name"],
        "pl_rade": mod_row["pl_rade"],
        "pl_bmasse": mod_row["pl_bmasse"],
        "pl_insol": mod_row["pl_insol"],
        "selected_threshold": mod_row["selected_threshold"],
        "label_habitable": mod_row["label_habitable"],
        "prompt": mod_prompt,
        "response": mod_response,
    })
    print("TABLE BEING SENT:")
    print(row_to_table(orig_row))
    print("---")
    print(row_to_table(mod_row))

    print(f"Finished pair_id={pair_id}")

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("qwen_results_part2_final.csv", index=False)
results_df.head()

In [ ]:
!nvidia-smi